In [2]:
import pandas as pd
import numpy as np
from functools import reduce

## 1. Выгружаем данные

In [4]:
df_births = pd.read_csv(r'G:\Демография\1. births.filtered\births.csv')
df_migration = pd.read_csv(r"G:\Демография\1.1 annual-net-migration.filtered\annual-net-migration.csv")
df_deaths = pd.read_csv(r"G:\Демография\2. deaths.filtered\deaths.csv")
df_infant_deaths = pd.read_csv(r"G:\Демография\2.1 number-of-infant-deaths.filtered\number-of-infant-deaths.csv")
df_child_deaths = pd.read_csv(r"G:\Демография\2.2 number-of-child-deaths.filtered\number-of-child-deaths.csv")
df_country_literacy = pd.read_csv(r"G:\Демография\3. cross-country-literacy-rates\cross-country-literacy-rates.csv")
df_military_of_gdp = pd.read_csv(r"G:\Демография\4. military-spending-as-a-share-of-gdp-sipri\military-spending-as-a-share-of-gdp-sipri.csv")
df_population_density = pd.read_csv(r"G:\Демография\5. population-density.filtered\population-density.csv")
df_population = pd.read_csv(r"G:\Демография\6. population\population.csv")
df_sex_ration = pd.read_csv(r"G:\Демография\7. sex-ratio.filtered\sex-ratio.csv")
df_fertility_rate = pd.read_csv(r"G:\Демография\1.2 fertility-rate-children-per-woman\fertility-rate-children-per-woman.csv")

In [5]:
data_frames = [
    df_births, 
    df_migration, 
    df_deaths, 
    df_infant_deaths, 
    df_child_deaths, 
    df_population_density,
    df_population,
    df_sex_ration,
    df_fertility_rate
]

## 2. Соединяем таблицы с одинаковыми столбцами

In [7]:
df_merged = reduce(lambda left, right: pd.merge(left, right, on=['Entity', 'Code', 'Year'], how='outer'), data_frames)

In [8]:
df_merged.head()

,Entity,Code,Year,"Births, total","Net migration, total","Number of deaths, total","Number of deaths, at birth","Number of deaths, ages 0-4","Population density, total",all years,"Sex ratio, total","Fertility rate, total"
0,Afghanistan,AFG,1950,383985.0,6161.0,290972.0,109220.0,160791.0,11.976,7776180.0,112.618350,7.248
1,Afghanistan,AFG,1951,391002.0,4903.0,288752.0,107971.0,158566.0,12.135,7879343.0,112.082695,7.260
2,Afghanistan,AFG,1952,397663.0,145.0,288059.0,108140.0,157832.0,12.302,7987784.0,111.549510,7.260
3,Afghanistan,AFG,1953,404666.0,-8867.0,287712.0,108248.0,157554.0,12.470,8096703.0,111.028630,7.266
4,Afghanistan,AFG,1954,410428.0,-6837.0,289189.0,108241.0,157723.0,12.641,8207954.0,110.527370,7.254


In [9]:
df_merged = df_merged.rename(columns={'all years': 'Population'})

### 3. Определяем Code_Types и создаем столбец под все виды

In [11]:
unique_code = df_merged['Code'].unique()

In [12]:
unique_code

array(['AFG', 'OWID_AFR', 'UN_AFR', nan, 'ALB', 'DZA', 'ASM', 'AND',
       'AGO', 'AIA', 'ATG', 'ARG', 'ARM', 'ABW', 'OWID_ASI', 'UN_ASI',
       'AUS', 'AUT', 'AZE', 'BHS', 'BHR', 'BGD', 'BRB', 'BLR', 'BEL',
       'BLZ', 'BEN', 'BMU', 'BTN', 'BOL', 'BES', 'BIH', 'BWA', 'BRA',
       'VGB', 'BRN', 'BGR', 'BFA', 'BDI', 'KHM', 'CMR', 'CAN', 'CPV',
       'CYM', 'CAF', 'TCD', 'CHL', 'CHN', 'COL', 'COM', 'COG', 'COK',
       'CRI', 'CIV', 'HRV', 'CUB', 'CUW', 'CYP', 'CZE', 'COD', 'DNK',
       'DJI', 'DMA', 'DOM', 'TLS', 'ECU', 'EGY', 'SLV', 'GNQ', 'ERI',
       'EST', 'SWZ', 'ETH', 'OWID_EUR', 'UN_EUR', 'FLK', 'FRO', 'FJI',
       'FIN', 'FRA', 'GUF', 'PYF', 'GAB', 'GMB', 'GEO', 'DEU', 'GHA',
       'GIB', 'GRC', 'GRL', 'GRD', 'GLP', 'GUM', 'GTM', 'GGY', 'GIN',
       'GNB', 'GUY', 'HTI', 'OWID_HIC', 'HND', 'HKG', 'HUN', 'ISL', 'IND',
       'IDN', 'IRN', 'IRQ', 'IRL', 'IMN', 'ISR', 'ITA', 'JAM', 'JPN',
       'JEY', 'JOR', 'KAZ', 'KEN', 'KIR', 'OWID_KOS', 'KWT', 'KGZ', 'LAO',
       'U

In [13]:
len(unique_code)

255

In [14]:
def classify_code(code):
    c = str(code)
    if c == 'OWID_WRL': # 1. Весь мир
        return 'World'
    elif c in ['OWID_HIC', 'OWID_LIC', 'OWID_LMC', 'OWID_UMC']: # 2. По ВВП 
        return 'Income Group'
    elif c.startswith('OWID_') and c not in ['OWID_KOS']: # 3. Континенты по версии OWID
        return 'Continent (OWID)'
    elif c.startswith('UN_'): # 4. Континенты по версии ООН
        return 'Region (UN)'
    elif len(c) == 3 and c.isupper(): # 5. Страны
        return 'Country'
    elif c == 'OWID_KOS':
        return 'Country' # Косово считаем как страну
    else: # Другое
        return 'Territory/Other'

### Список Income Group: 
- OWID_HIC (High Income — Высокий доход)
- OWID_UMC (Upper Middle Income — Выше среднего)
- OWID_LMC (Lower Middle Income — Ниже среднего)
- OWID_LIC (Low Income — Низкий доход)

In [16]:
df_merged['Code_Type'] = df_merged['Code'].apply(classify_code)

In [17]:
df_merged.head()

,Entity,Code,Year,"Births, total","Net migration, total","Number of deaths, total","Number of deaths, at birth","Number of deaths, ages 0-4","Population density, total",Population,"Sex ratio, total","Fertility rate, total",Code_Type
0,Afghanistan,AFG,1950,383985.0,6161.0,290972.0,109220.0,160791.0,11.976,7776180.0,112.618350,7.248,Country
1,Afghanistan,AFG,1951,391002.0,4903.0,288752.0,107971.0,158566.0,12.135,7879343.0,112.082695,7.260,Country
2,Afghanistan,AFG,1952,397663.0,145.0,288059.0,108140.0,157832.0,12.302,7987784.0,111.549510,7.260,Country
3,Afghanistan,AFG,1953,404666.0,-8867.0,287712.0,108248.0,157554.0,12.470,8096703.0,111.028630,7.266,Country
4,Afghanistan,AFG,1954,410428.0,-6837.0,289189.0,108241.0,157723.0,12.641,8207954.0,110.527370,7.254,Country


### 4. Определяем страны, которые не публикуют расходы на военку

In [19]:
df_military_of_gdp.head()

,Entity,Code,Year,Military expenditure (% of GDP),World region according to OWID
0,Afghanistan,AFG,1970,1.629606,Asia
1,Afghanistan,AFG,1973,1.868910,Asia
2,Afghanistan,AFG,1974,1.610825,Asia
3,Afghanistan,AFG,1975,1.722066,Asia
4,Afghanistan,AFG,1976,2.046087,Asia


In [20]:
region_map = df_military_of_gdp[['Entity', 'World region according to OWID']].drop_duplicates().dropna()

In [21]:
region_map.info

<bound method DataFrame.info of            Entity World region according to OWID
0     Afghanistan                           Asia
24        Albania                         Europe
69        Algeria                         Africa
133        Angola                         Africa
173     Argentina                  South America
...           ...                            ...
8034    Venezuela                  South America
8103      Vietnam                           Asia
8127        Yemen                           Asia
8152       Zambia                         Africa
8188     Zimbabwe                         Africa

[165 rows x 2 columns]>

In [22]:
all_countries = set(df_merged[df_merged['Code_Type'] == 'Country']['Entity'].unique())

military_entities = set(df_military_of_gdp['Entity'].unique())

missing_countries = sorted(list(all_countries - military_entities))

In [23]:
len(missing_countries)

72

In [24]:
missing_countries

['American Samoa',
 'Andorra',
 'Anguilla',
 'Antigua and Barbuda',
 'Aruba',
 'Bahamas',
 'Barbados',
 'Bermuda',
 'Bhutan',
 'Bonaire Sint Eustatius and Saba',
 'British Virgin Islands',
 'Cayman Islands',
 'Comoros',
 'Cook Islands',
 'Costa Rica',
 'Curacao',
 'Dominica',
 'Falkland Islands',
 'Faroe Islands',
 'French Guiana',
 'French Polynesia',
 'Gibraltar',
 'Greenland',
 'Grenada',
 'Guadeloupe',
 'Guam',
 'Guernsey',
 'Hong Kong',
 'Iceland',
 'Isle of Man',
 'Jersey',
 'Kiribati',
 'Liechtenstein',
 'Macao',
 'Maldives',
 'Marshall Islands',
 'Martinique',
 'Mayotte',
 'Micronesia (country)',
 'Monaco',
 'Montserrat',
 'Nauru',
 'New Caledonia',
 'Niue',
 'North Korea',
 'Northern Mariana Islands',
 'Palau',
 'Palestine',
 'Puerto Rico',
 'Reunion',
 'Saint Barthelemy',
 'Saint Helena',
 'Saint Kitts and Nevis',
 'Saint Lucia',
 'Saint Martin (French part)',
 'Saint Pierre and Miquelon',
 'Saint Vincent and the Grenadines',
 'Samoa',
 'San Marino',
 'Sao Tome and Principe',

### 5. Создаем столбец Wotld_Region (Части света)

In [26]:
full_region_map = dict(zip(region_map['Entity'], region_map['World region according to OWID']))

In [27]:
manual_regions = {
    # Европа
    'Andorra': 'Europe', 'Faroe Islands': 'Europe', 'Gibraltar': 'Europe', 'Guernsey': 'Europe', 
    'Iceland': 'Europe', 'Isle of Man': 'Europe', 'Jersey': 'Europe', 'Liechtenstein': 'Europe', 
    'Monaco': 'Europe', 'San Marino': 'Europe', 'Vatican': 'Europe',
    
    # Азия
    'Bhutan': 'Asia', 'Hong Kong': 'Asia', 'Macao': 'Asia', 'Maldives': 'Asia', 
    'North Korea': 'Asia', 'Palestine': 'Asia',
    
    # Северная Америка (включая Карибы)
    'Anguilla': 'North America', 'Antigua and Barbuda': 'North America', 'Aruba': 'North America', 
    'Bahamas': 'North America', 'Barbados': 'North America', 'Bermuda': 'North America', 
    'British Virgin Islands': 'North America', 'Cayman Islands': 'North America', 
    'Costa Rica': 'North America', 'Dominica': 'North America', 'Greenland': 'North America', 
    'Grenada': 'North America', 'Puerto Rico': 'North America', 'Saint Kitts and Nevis': 'North America', 
    'Saint Lucia': 'North America', 'Saint Vincent and the Grenadines': 'North America', 
    'Turks and Caicos Islands': 'North America', 'United States Virgin Islands': 'North America',
    
    # Океания
    'American Samoa': 'Oceania', 'Cook Islands': 'Oceania', 'French Polynesia': 'Oceania', 
    'Guam': 'Oceania', 'Kiribati': 'Oceania', 'Marshall Islands': 'Oceania', 'Nauru': 'Oceania', 
    'New Caledonia': 'Oceania', 'Palau': 'Oceania', 'Samoa': 'Oceania', 'Solomon Islands': 'Oceania', 
    'Tonga': 'Oceania', 'Tuvalu': 'Oceania', 'Vanuatu': 'Oceania',
    
    # Южная Америка
    'Suriname': 'South America', 'French Guiana': 'South America'
}

In [28]:
full_region_map.update(manual_regions)

In [29]:
df_merged['World_Region'] = df_merged['Entity'].map(full_region_map)

### Проверка стран в выборке

In [31]:
countries_list = sorted(df_merged[df_merged['Code_Type'] == 'Country']['Entity'].unique())

In [32]:
len(countries_list)

237

In [33]:
countries_list

['Afghanistan',
 'Albania',
 'Algeria',
 'American Samoa',
 'Andorra',
 'Angola',
 'Anguilla',
 'Antigua and Barbuda',
 'Argentina',
 'Armenia',
 'Aruba',
 'Australia',
 'Austria',
 'Azerbaijan',
 'Bahamas',
 'Bahrain',
 'Bangladesh',
 'Barbados',
 'Belarus',
 'Belgium',
 'Belize',
 'Benin',
 'Bermuda',
 'Bhutan',
 'Bolivia',
 'Bonaire Sint Eustatius and Saba',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'British Virgin Islands',
 'Brunei',
 'Bulgaria',
 'Burkina Faso',
 'Burundi',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Cape Verde',
 'Cayman Islands',
 'Central African Republic',
 'Chad',
 'Chile',
 'China',
 'Colombia',
 'Comoros',
 'Congo',
 'Cook Islands',
 'Costa Rica',
 "Cote d'Ivoire",
 'Croatia',
 'Cuba',
 'Curacao',
 'Cyprus',
 'Czechia',
 'Democratic Republic of Congo',
 'Denmark',
 'Djibouti',
 'Dominica',
 'Dominican Republic',
 'East Timor',
 'Ecuador',
 'Egypt',
 'El Salvador',
 'Equatorial Guinea',
 'Eritrea',
 'Estonia',
 'Eswatini',
 'Ethiopia',
 'Falkland Islan

### 6. Создаем столбец наличия данных по военным расходам

In [35]:
df_merged['has_military_data'] = df_merged['Entity'].isin(military_entities).astype(int)
# df_merged['has_military_data'] = if df_merged['Entity'] == military_entities: 1, else 0

In [36]:
df_merged[df_merged['Entity'] == 'Russia'][df_merged['has_military_data']!=0]

C:\Users\ilyal\AppData\Local\Temp\ipykernel_3388\74883587.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_merged[df_merged['Entity'] == 'Russia'][df_merged['has_military_data']!=0]


,Entity,Code,Year,"Births, total","Net migration, total","Number of deaths, total","Number of deaths, at birth","Number of deaths, ages 0-4","Population density, total",Population,"Sex ratio, total","Fertility rate, total",Code_Type,World_Region,has_military_data
15096,Russia,RUS,1950,2884600.0,-69764.0,1363062.0,322569.0,465206.0,6.314,103392364.0,76.905350,2.926,Country,Europe,1
15097,Russia,RUS,1951,2920639.0,-84687.0,1383142.0,337255.0,496784.0,6.402,104844658.0,77.353230,2.902,Country,Europe,1
15098,Russia,RUS,1952,2928571.0,-128136.0,1289801.0,283668.0,403464.0,6.493,106326385.0,77.797320,2.863,Country,Europe,1
15099,Russia,RUS,1953,2922390.0,-136677.0,1245750.0,258767.0,364788.0,6.586,107851682.0,78.236790,2.829,Country,Europe,1
15100,Russia,RUS,1954,2946027.0,-64245.0,1198621.0,264453.0,368350.0,6.685,109463245.0,78.672905,2.852,Country,Europe,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15165,Russia,RUS,2019,1496027.0,336494.0,1789415.0,7175.0,9251.0,8.948,146533063.0,86.692770,1.504,Country,Europe,1
15166,Russia,RUS,2020,1458020.0,334094.0,2158768.0,6732.0,8660.0,8.938,146371295.0,86.707580,1.505,Country,Europe,1
15167,Russia,RUS,2021,1421809.0,314112.0,2439514.0,6372.0,8445.0,8.906,145836172.0,86.756226,1.505,Country,Europe,1
15168,Russia,RUS,2022,1334912.0,765629.0,1909498.0,5237.0,6816.0,8.890,145579889.0,86.762020,1.452,Country,Europe,1


### 7. Создаем столбец population_type

In [38]:
latest_population = df_merged[df_merged['Year'] == 2021][['Entity', 'Population']]

def classify_population(population):
    if population > 500000000: return '1. Global Giants (>0.5В)'
    elif population > 250000000: return '2. Super Large (250M-500M)'
    elif population > 100000000:  return '3. Large (100M-250M)'
    elif population > 50000000:  return '4. Medium (50M-100M)'
    elif population > 20000000:  return '5. Low Medium (20M-50M)'
    elif population > 5000000:  return '6. Small (5M-20M)'
    else: return '7. Really Small (<5M)'

latest_population['Population_Type'] = latest_population['Population'].apply(classify_population)

df_merged = pd.merge(df_merged, latest_population[['Entity', 'Population_Type']], on='Entity', how='left')

### 8. Создаем столбцы Population_Net_Change и Natural_Increase

In [40]:
df_merged['Population_Net_Change'] = (
    df_merged['Births, total'] - 
    df_merged['Number of deaths, total'] + 
    df_merged['Net migration, total']
)

# Для наглядности можно также посчитать "Естественный прирост" (без миграции)
df_merged['Natural_Increase'] = df_merged['Births, total'] - df_merged['Number of deaths, total']

## 9. Обработка пропусков

In [42]:
df_merged.isnull().sum()

Entity                           0
Code                          2072
Year                             0
Births, total                 1702
Net migration, total          1702
Number of deaths, total       1924
Number of deaths, at birth    1924
Number of deaths, ages 0-4    1924
Population density, total     2146
Population                    1480
Sex ratio, total              1924
Fertility rate, total         1702
Code_Type                        0
World_Region                  5624
has_military_data                0
Population_Type                  0
Population_Net_Change         3626
Natural_Increase              3626
dtype: int64

In [43]:
df_merged.shape

(21608, 18)

In [44]:
df_final = df_merged.copy()

In [45]:
df_final['Code'] = df_final['Code'].fillna('N/A')
df_final['World_Region'] = df_final['World_Region'].fillna('Other/Aggregates')

In [46]:
mask = df_final['Population_Net_Change'].isna() & df_final['Births, total'].notna() & df_final['Number of deaths, total'].notna()
df_final.loc[mask, 'Population_Net_Change'] = (
    df_final['Births, total'] - 
    df_final['Number of deaths, total'] + 
    df_final['Net migration, total'].fillna(0)
)

### Дорабатываем и выгружаем df

In [47]:
df_final['Date'] = pd.to_datetime(df_final['Year'], format='%Y')

In [48]:
cols_to_fix = [
    'Births, total', 'Net migration, total', 'Number of deaths, total', 
    'Number of deaths, at birth', 'Number of deaths, ages 0-4', 
    'Population density, total', 'Population',
    'Sex ratio, total', 'Fertility rate, total',
    'Population_Net_Change', 'Natural_Increase'
]

for col in cols_to_fix:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

In [49]:
df_final.isnull().sum()

Entity                           0
Code                             0
Year                             0
Births, total                 1702
Net migration, total          1702
Number of deaths, total       1924
Number of deaths, at birth    1924
Number of deaths, ages 0-4    1924
Population density, total     2146
Population                    1480
Sex ratio, total              1924
Fertility rate, total         1702
Code_Type                        0
World_Region                     0
has_military_data                0
Population_Type                  0
Population_Net_Change         3626
Natural_Increase              3626
Date                             0
dtype: int64

In [50]:
df_final.rename(columns={
    'Births, total': 'Births_Total',
    'Net migration, total': 'Net_Migration_Total',
    'Number of deaths, total': 'Deaths_Total',
    'Number of deaths, at birth': 'Deaths_At_Birth',
    'Number of deaths, ages 0-4': 'Deaths_Ages_0_4',
    'Population density, total': 'Population_Density_Total',
    'Sex ratio, total': 'Sex_Ratio_Total',
    'Population, total': 'Population_Total',
    'Fertility rate, total': 'Fertility_Rate_Total'
}, inplace=True)

In [51]:
df_final = df_final[df_final['Code_Type'] == 'Country'].copy()

df_final.drop(columns=['Code_Type'], inplace=True, errors='ignore')

In [52]:
df_final.to_csv(r"G:\Демография\prepared_data.csv", index=False, encoding='utf-8-sig')